# 4 · JAX

`duq.jax.Quantity` couples a JAX-array magnitude with a **static** unit. Units are checked and propagated at **trace time** via quax primitive interception, so compiled kernels pay zero runtime cost for units.

Requires the `duq[jax]` extra.

In [1]:
import jax
import duq, duq.jax
import duq.jax.numpy as djnp

q = duq.jax.Quantity([1.0, 2.0, 3.0], "m")
print(q)
print(q.to("cm").value)

[1. 2. 3.] m


[100. 200. 300.]


## Operators, `duq.jax.numpy`, and `.at`

In [2]:
print((q + q).value)
print(str(djnp.sqrt(q * q).unit))
print(q.at[0].set(duq.jax.Quantity(500.0, "cm")).value)   # update converted to m

[2. 4. 6.]


m


[5. 2. 3.]


## Transformations

A quantity is a pytree, so `jax.jit`/`jax.vmap` accept it directly; `duq.jax.grad`/`hessian` label derivatives with `unit_out / unit_in`.

In [3]:
print(jax.jit(lambda x: x * x)(q).value)

def energy(x):                 # spring: E = 1/2 k x^2
    return 0.5 * duq.jax.Quantity(2.0, "N/m") * x * x

g = duq.jax.grad(energy)(duq.jax.Quantity(0.1, "m"))
h = duq.jax.hessian(energy)(duq.jax.Quantity(0.1, "m"))
print(g.value, g.unit)   # force
print(h.value, h.unit)   # stiffness

[1. 4. 9.]


0.2 N
2.0 N·m⁻¹


## Bare operands: a NumPy ↔ JAX difference

In selection/update ops (`where`, `pad`, `scatter`, `dynamic_update_slice`) that mix a quantity with a **bare (unit-less)** operand, the two back-ends differ **by design**:

* the **NumPy** layer is strict — a bare operand raises `DimensionalityError` (it never silently acquires a unit);
* the **`duq.jax`** layer lets the bare operand **adopt** the quantity's unit.

This is a constraint of primitive-level interception: after tracing, a `jit`-internal literal is indistinguishable from a user-provided bare array, so the JAX layer cannot reject bare operands without breaking legitimate compiled code (`unxt` behaves the same). Wrap the operand in a `Quantity` when you want the strict check.

In [4]:
import numpy as np

# NumPy: strict — the bare [3., 4.] has no unit.
np.where([True, False], duq.Quantity(np.array([1.0, 2.0]), "m"), np.array([3.0, 4.0]))

DimensionalityError: cannot combine a bare array/number with a quantity of dimension L

In [5]:
import jax.numpy as jnp

# duq.jax: the bare [3., 4.] adopts metres.
djnp.where(jnp.array([True, False]),
           duq.jax.Quantity(jnp.array([1.0, 2.0]), "m"),
           jnp.array([3.0, 4.0]))

Quantity(Array([1., 4.], dtype=float32), Unit('m'))

## Hot kernels

The unit is part of `jit`'s cache key (metres then kilometres compiles twice). For hot kernels, strip to a canonical unit **before** the kernel.

In [6]:
mag = duq.ustrip("m", q)          # a plain jax.Array, no unit bookkeeping
out = duq.jax.Quantity(mag * 2.0, "m/s")
out

Quantity(Array([2., 4., 6.], dtype=float32), Unit('m.s^-1'))